# AI-Powered Healthcare Risk Prediction and Monitoring System
**Google Colab – complete single-notebook project**

Modules: Diabetes prediction, heart disease prediction, dataset explorer, vital-sign monitoring, PDF report, and RAG healthcare assistant.

> **Important:** This is an educational screening prototype, not a medical device. Predictions and threshold alerts are not diagnoses. Do not use them to make treatment decisions. The sample datasets are public benchmark datasets and may not represent every population.

## API setup
The RAG assistant uses an **OpenAI-compatible chat-completions API adapter**. Since the provider and exact Nova model name are not known yet, fill in `API_BASE_URL` and `MODEL_NAME` from your provider's documentation. Keep your API key private; enter it in the Colab secret prompt when the notebook runs. If your provider does not support the OpenAI-compatible API format, its SDK/request format must be adapted.


In [ ]:
# 1. Install required packages
!pip -q install gradio scikit-learn pandas numpy matplotlib seaborn reportlab openai requests


In [ ]:
# 2. Imports and configuration
import os, io, json, warnings, textwrap
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# Configure these using your API provider's official documentation.
# Example format only: https://your-provider.example/v1
API_BASE_URL = ""       # Required for live RAG answers
MODEL_NAME = ""         # Exact model identifier from your provider
API_KEY = ""

# Securely ask for the key at runtime. Do not paste it into a shared notebook.
try:
    from google.colab import userdata
    API_KEY = userdata.get("NOVA_API_KEY") or ""
except Exception:
    pass

if not API_KEY:
    import getpass
    API_KEY = getpass.getpass("Enter your API key (input hidden; leave blank to use retrieval-only mode): ")

print("Configuration loaded. API key present:", bool(API_KEY))


## 3. Load public datasets
The notebook attempts to download public Pima diabetes and Cleveland heart-disease CSV datasets. If a source is unavailable, upload your own CSV files in the next cell. The expected target column is `Outcome` for diabetes and `target` (or `num`) for heart disease.

In [ ]:
# 3A. Download benchmark datasets
import requests

DIABETES_URL = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
HEART_URL = "https://raw.githubusercontent.com/plotly/datasets/master/heart.csv"

def get_csv(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return pd.read_csv(io.BytesIO(r.content))

diabetes_df, heart_df = None, None
try:
    diabetes_df = get_csv(DIABETES_URL)
    print("Diabetes dataset:", diabetes_df.shape, list(diabetes_df.columns))
except Exception as e:
    print("Could not download diabetes dataset:", e)

try:
    heart_df = get_csv(HEART_URL)
    print("Heart dataset:", heart_df.shape, list(heart_df.columns))
except Exception as e:
    print("Could not download heart dataset:", e)



In [ ]:
# 3B. Optional CSV upload if a download failed or you have approved datasets
try:
    from google.colab import files
    if diabetes_df is None or heart_df is None:
        print("Upload the missing CSV file(s).")
        uploaded = files.upload()
        for fname, content in uploaded.items():
            df = pd.read_csv(io.BytesIO(content))
            low = fname.lower()
            if "diab" in low:
                diabetes_df = df
            elif "heart" in low:
                heart_df = df
except Exception:
    pass

if diabetes_df is None or heart_df is None:
    raise RuntimeError("A dataset is missing. Upload diabetes and heart CSV files, then rerun this cell.")
display(diabetes_df.head())
display(heart_df.head())


## 4. Prepare data and train models
The target columns are detected and standardized. Multiple classifiers are compared using a held-out test split. Metrics are for learning/demo purposes and are not clinical validation.

In [ ]:
# 4A. Target detection and training utilities
def detect_target(df, candidates):
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    raise ValueError(f"Target column not found. Expected one of {candidates}; found {list(df.columns)}")

def clean_binary_target(y):
    # Common Cleveland heart dataset: 0 = no disease, 1-4 = disease
    vals = pd.Series(y).dropna().unique()
    if len(vals) > 2:
        return (pd.to_numeric(y, errors="coerce") > 0).astype(int)
    # Normalize text labels where possible
    if pd.Series(y).dtype == "object":
        s = pd.Series(y).astype(str).str.lower().str.strip()
        positive = {"1","yes","true","positive","disease","present"}
        return s.isin(positive).astype(int)
    return pd.to_numeric(y, errors="coerce").fillna(0).astype(int)

def train_task(df, target_candidates, task_name):
    target = detect_target(df, target_candidates)
    data = df.copy()
    y = clean_binary_target(data.pop(target))
    # Drop rows with missing labels and remove duplicate rows
    valid = y.notna()
    data, y = data.loc[valid].copy(), y.loc[valid]
    data = data.drop_duplicates()
    y = y.loc[data.index]
    # Convert numeric-looking columns
    for c in data.columns:
        data[c] = pd.to_numeric(data[c], errors="ignore")
    numeric = data.select_dtypes(include=np.number).columns.tolist()
    categorical = [c for c in data.columns if c not in numeric]
    transformers = []
    if numeric:
        transformers.append(("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric))
    if categorical:
        transformers.append(("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical))
    prep = ColumnTransformer(transformers)
    candidates = {
        "Logistic Regression": LogisticRegression(max_iter=1500, class_weight="balanced"),
        "Random Forest": RandomForestClassifier(n_estimators=250, random_state=RANDOM_STATE, class_weight="balanced"),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE)
    }
    stratify = y if y.nunique() == 2 and y.value_counts().min() >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(data, y, test_size=0.2, random_state=RANDOM_STATE, stratify=stratify)
    results, fitted = [], {}
    for name, estimator in candidates.items():
        pipe = Pipeline([("prep", prep), ("model", estimator)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        try:
            proba = pipe.predict_proba(X_test)[:,1]
            auc = roc_auc_score(y_test, proba)
        except Exception:
            auc = np.nan
        acc = accuracy_score(y_test, pred)
        results.append({"Model": name, "Accuracy": round(acc,4), "ROC-AUC": round(float(auc),4) if pd.notna(auc) else None})
        fitted[name] = pipe
    result_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False, na_position="last")
    best_name = result_df.iloc[0]["Model"]
    print(f"{task_name}: target={target}; best model={best_name}")
    display(result_df)
    return {"model": fitted[best_name], "all_models": fitted, "metrics": result_df,
            "features": data.columns.tolist(), "target": target, "X_test": X_test, "y_test": y_test}

diabetes_task = train_task(diabetes_df, ["Outcome", "outcome", "class"], "Diabetes")
heart_task = train_task(heart_df, ["target", "num", "condition", "output"], "Heart disease")


## 5. Data Explorer
Inspect dataset size, columns, missing values, class balance and model comparison.

In [ ]:
def explore_dataset(which):
    df = diabetes_df if which == "Diabetes" else heart_df
    task = diabetes_task if which == "Diabetes" else heart_task
    summary = pd.DataFrame({
        "Column": df.columns,
        "Data type": [str(x) for x in df.dtypes],
        "Missing values": [int(x) for x in df.isna().sum()]
    })
    return f"Rows: {len(df)} | Columns: {len(df.columns)}", summary, task["metrics"], df.head(10)



## 6. Prediction helpers
Inputs are collected dynamically from the selected dataset's feature columns. Numeric features accept numbers; categorical features accept text. The UI uses a basic input form and reports a model-estimated class probability.

In [ ]:
def make_input_fields(task):
    fields = []
    for col in task["features"]:
        series = (diabetes_df if task is diabetes_task else heart_df)[col]
        if pd.api.types.is_numeric_dtype(series):
            val = float(pd.to_numeric(series, errors="coerce").median())
            fields.append((col, val, "number"))
        else:
            mode = series.dropna().mode()
            val = str(mode.iloc[0]) if len(mode) else ""
            fields.append((col, val, "text"))
    return fields

def predict_task(task, df, values):
    row = {}
    for col in task["features"]:
        raw = values.get(col, np.nan)
        if pd.api.types.is_numeric_dtype(df[col]):
            try: row[col] = float(raw)
            except: row[col] = np.nan
        else:
            row[col] = str(raw)
    x = pd.DataFrame([row], columns=task["features"])
    model = task["model"]
    pred = int(model.predict(x)[0])
    try: probability = float(model.predict_proba(x)[0][1])
    except: probability = None
    label = "Higher model-estimated risk class" if pred == 1 else "Lower model-estimated risk class"
    prob_text = f" Estimated positive-class probability: {probability:.1%}." if probability is not None else ""
    return f"**{label}.**{prob_text}\n\nThis is an educational model output, not a diagnosis. Please discuss health concerns with a qualified clinician."



## 7. Vital-sign monitoring
The demo uses editable illustrative thresholds. These are not personalized medical thresholds. A user should follow clinician-provided ranges.

In [ ]:
def monitor_vitals(temperature, heart_rate, systolic_bp, diastolic_bp, spo2, glucose):
    checks = [
        ("Temperature", temperature, 36.0, 37.5, "°C"),
        ("Heart rate", heart_rate, 60, 100, "bpm"),
        ("Systolic blood pressure", systolic_bp, 90, 120, "mmHg"),
        ("Diastolic blood pressure", diastolic_bp, 60, 80, "mmHg"),
        ("SpO₂", spo2, 95, 100, "%"),
        ("Glucose (fasting)", glucose, 70, 99, "mg/dL")
    ]
    lines=[]
    for name, value, low, high, unit in checks:
        status = "Within demo range" if low <= value <= high else "Outside demo range – seek appropriate clinical advice"
        lines.append(f"- {name}: {value} {unit} — {status} (illustrative range {low}–{high} {unit})")
    return "\n".join(lines) + "\n\nThese illustrative ranges are not a diagnostic tool or emergency triage system. If you feel seriously unwell, seek urgent medical care."


## 8. PDF report generation
Creates a downloadable summary of the user's entered measurements and model output. Avoid including unnecessary personal identifiers.

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

def create_pdf_report(name, diabetes_result, heart_result, monitoring_result):
    buffer = io.BytesIO()
    doc = SimpleDocTemplate(buffer, pagesize=A4)
    styles = getSampleStyleSheet()
    story = [Paragraph("AI-Powered Healthcare Risk Prediction – Educational Report", styles["Title"]),
             Spacer(1, 12),
             Paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}", styles["Normal"]),
             Paragraph(f"Name/label: {name or 'Not provided'}", styles["Normal"]), Spacer(1, 12)]
    for heading, content in [("Diabetes model output", diabetes_result), ("Heart disease model output", heart_result), ("Vital-sign monitoring", monitoring_result)]:
        story.append(Paragraph(heading, styles["Heading2"]))
        story.append(Paragraph((content or "Not provided").replace("\n","<br/>"), styles["BodyText"]))
        story.append(Spacer(1, 10))
    story.append(Paragraph("<b>Disclaimer:</b> Educational prototype only. This report is not a medical diagnosis or treatment recommendation. Consult a qualified healthcare professional.", styles["BodyText"]))
    doc.build(story)
    buffer.seek(0)
    path = "/content/healthcare_risk_report.pdf"
    with open(path, "wb") as f: f.write(buffer.read())
    return path


## 9. RAG healthcare assistant (retrieval + Nova/API)
This demo indexes short educational notes, retrieves relevant chunks using TF-IDF, then optionally sends the retrieved context to your configured OpenAI-compatible API. Without a working API configuration, it returns retrieved source notes only. Replace or extend the knowledge base with vetted, licensed healthcare references before any real-world use.

In [ ]:
# Compact starter knowledge base – educational only; not a substitute for clinical guidance.
KNOWLEDGE = [
("Diabetes overview", "Diabetes is a condition involving elevated blood glucose. Risk factors can include family history, age, body weight, and physical inactivity. Screening and diagnosis require appropriate clinical tests interpreted by a healthcare professional."),
("Diabetes tests", "Common tests used by clinicians include fasting plasma glucose, HbA1c, and oral glucose tolerance testing. A single app prediction cannot diagnose diabetes."),
("Heart disease overview", "Heart disease includes several conditions affecting the heart and blood vessels. Risk factors can include high blood pressure, high cholesterol, smoking, diabetes, age, and family history."),
("Heart symptoms", "Chest discomfort, shortness of breath, fainting, or sudden severe symptoms can require urgent medical assessment. Do not rely on a chatbot or prediction score to assess an emergency."),
("Blood pressure", "Blood pressure readings vary. Proper measurement technique and repeated readings matter. A clinician should interpret readings in the context of the person's health and history."),
("Healthy habits", "General heart and metabolic health can benefit from balanced nutrition, regular physical activity appropriate to ability, avoiding tobacco, adequate sleep, and routine preventive care."),
("Medical advice", "For personal diagnosis, medication, or treatment decisions, consult a qualified healthcare professional who can review symptoms, examination findings, and appropriate tests.")
]
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
kb_texts = [title + ". " + body for title, body in KNOWLEDGE]
vectorizer = TfidfVectorizer(stop_words="english")
kb_matrix = vectorizer.fit_transform(kb_texts)

def retrieve_context(question, k=3):
    qv = vectorizer.transform([question])
    scores = cosine_similarity(qv, kb_matrix).ravel()
    ids = np.argsort(scores)[::-1][:k]
    return [(KNOWLEDGE[i][0], KNOWLEDGE[i][1], float(scores[i])) for i in ids]

def rag_answer(question):
    if not question.strip():
        return "Please enter a healthcare question."
    retrieved = retrieve_context(question)
    context = "\n".join([f"[{title}] {body}" for title, body, score in retrieved])
    citations = "\n".join([f"- {title}" for title, body, score in retrieved])
    if API_KEY and API_BASE_URL and MODEL_NAME:
        try:
            from openai import OpenAI
            client = OpenAI(api_key=API_KEY, base_url=API_BASE_URL)
            response = client.chat.completions.create(
                model=MODEL_NAME,
                temperature=0.2,
                messages=[
                    {"role":"system","content":"You are an educational healthcare information assistant. Answer only from the supplied context. If the context is insufficient, say so. Do not diagnose, prescribe, or give emergency triage. Encourage a qualified clinician for personal concerns."},
                    {"role":"user","content":f"Question: {question}\n\nRetrieved context:\n{context}\n\nGive a concise answer and mention the source-note titles used."}
                ]
            )
            answer = response.choices[0].message.content
            return answer + "\n\nRetrieved notes:\n" + citations
        except Exception as e:
            return "API call failed. Check API_BASE_URL, MODEL_NAME, key, and provider compatibility.\n\nRetrieved context (fallback):\n" + context + "\n\nError: " + str(e)
    return "API is not configured, so this is retrieval-only mode.\n\nRelevant notes:\n" + context + "\n\nRetrieved notes:\n" + citations + "\n\nAdd your provider's base URL and exact model ID to enable generated answers."


## 10. Build the Gradio web application
The app exposes the six project modules. Prediction fields are generated from the dataset columns; the demo dataset feature names and units may differ from clinical input forms.

In [ ]:
# Create dynamic Gradio input components
def gradio_components(task, df):
    comps=[]
    for col in task["features"]:
        if pd.api.types.is_numeric_dtype(df[col]):
            default=float(pd.to_numeric(df[col], errors="coerce").median())
            comps.append(gr.Number(label=col, value=default))
        else:
            mode=df[col].dropna().mode()
            default=str(mode.iloc[0]) if len(mode) else ""
            comps.append(gr.Textbox(label=col, value=default))
    return comps

diab_inputs = gradio_components(diabetes_task, diabetes_df)
heart_inputs = gradio_components(heart_task, heart_df)

def diabetes_ui(*vals):
    return predict_task(diabetes_task, diabetes_df, dict(zip(diabetes_task["features"], vals)))
def heart_ui(*vals):
    return predict_task(heart_task, heart_df, dict(zip(heart_task["features"], vals)))

with gr.Blocks(title="AI-Powered Healthcare Risk Prediction") as demo:
    gr.Markdown("# AI-Powered Healthcare Risk Prediction and Monitoring System")
    gr.Markdown("Educational prototype only — not a medical diagnosis device.")
    with gr.Tab("Diabetes Prediction"):
        gr.Markdown("Enter values matching the diabetes dataset features.")
        gradio_diabetes = diab_inputs
        diab_btn = gr.Button("Predict diabetes risk class")
        diab_out = gr.Markdown()
        diab_btn.click(diabetes_ui, inputs=gradio_diabetes, outputs=diab_out)
    with gr.Tab("Heart Disease Prediction"):
        gr.Markdown("Enter values matching the heart dataset features.")
        heart_btn = gr.Button("Predict heart disease risk class")
        heart_out = gr.Markdown()
        heart_btn.click(heart_ui, inputs=heart_inputs, outputs=heart_out)
    with gr.Tab("Data Explorer"):
        ds_choice = gr.Radio(["Diabetes","Heart Disease"], value="Diabetes", label="Choose dataset")
        ds_summary = gr.Textbox(label="Dataset summary")
        ds_missing = gr.Dataframe(label="Columns and missing values")
        ds_models = gr.Dataframe(label="Model comparison")
        ds_preview = gr.Dataframe(label="First 10 rows")
        ds_choice.change(explore_dataset, inputs=ds_choice, outputs=[ds_summary,ds_missing,ds_models,ds_preview])
        demo.load(explore_dataset, inputs=ds_choice, outputs=[ds_summary,ds_missing,ds_models,ds_preview])
    with gr.Tab("Vital Monitoring"):
        temp = gr.Number(label="Temperature (°C)", value=36.8)
        hr = gr.Number(label="Heart rate (bpm)", value=75)
        sbp = gr.Number(label="Systolic BP (mmHg)", value=115)
        dbp = gr.Number(label="Diastolic BP (mmHg)", value=75)
        spo2 = gr.Number(label="SpO₂ (%)", value=98)
        glucose = gr.Number(label="Fasting glucose (mg/dL)", value=90)
        monitor_btn = gr.Button("Check demo thresholds")
        monitor_out = gr.Markdown()
        monitor_btn.click(monitor_vitals, [temp,hr,sbp,dbp,spo2,glucose], monitor_out)
    with gr.Tab("PDF Report"):
        report_name = gr.Textbox(label="Name or report label (optional)")
        report_diab = gr.Textbox(label="Diabetes result (paste output)", lines=3)
        report_heart = gr.Textbox(label="Heart result (paste output)", lines=3)
        report_monitor = gr.Textbox(label="Monitoring result (paste output)", lines=4)
        pdf_btn = gr.Button("Generate PDF report")
        pdf_file = gr.File(label="Download report")
        pdf_btn.click(create_pdf_report, [report_name,report_diab,report_heart,report_monitor], pdf_file)
    with gr.Tab("RAG Healthcare Assistant"):
        question = gr.Textbox(label="Ask a general healthcare question", lines=3, placeholder="What is HbA1c?")
        ask_btn = gr.Button("Ask assistant")
        answer = gr.Markdown()
        ask_btn.click(rag_answer, question, answer)

print("Gradio app is ready.")


In [ ]:
# 11. Launch the app and create a public share link
# Keep this cell running while using the app. Gradio generates a temporary public URL.
demo.launch(share=True, debug=True)


## 12. Before your project review
1. Run the notebook from top to bottom in Google Colab.
2. Check the printed dataset columns and model metrics.
3. For live Nova/API answers, fill `API_BASE_URL` and `MODEL_NAME` in Section 2 using your provider's documentation, then rerun from Section 2 onward.
4. Store the API key in Colab Secrets as `NOVA_API_KEY` if possible. Never share the key or publish it in a notebook.
5. Test each tab and verify the generated PDF.
6. The Gradio share URL is temporary and works only while the Colab runtime remains active.

### Limitations
- The included datasets are benchmark datasets; model performance does not establish clinical safety.
- The starter RAG knowledge base is small and educational. Use reviewed, reliable medical references for a stronger project.
- The API adapter assumes OpenAI-compatible chat completions. If your provider uses another format, supply its provider name or API documentation so the request code can be adapted.
- No patient records are stored by this notebook. Avoid entering identifiable or sensitive health information.
